# SmolVLM 256M experiment record
Executed post-training scoring; optional fresh inference and export are distinct modes.

Original: `V3_EdgeVLM_256M_MediaPipe_PostTraining_Results.ipynb`. Numeric/text outputs are retained where present. Embedded visual outputs and image-path previews are removed for publication. Execution counts are not fabricated. See README.md and the source manifest.

This is a historical server notebook, not a portable CLI. Review paths, dependencies, adapter initialization and optional export cells before executing. No dataset or weights are included.


# MediaPipe SmolVLM-256M: post-training evaluation and results

This notebook **does not train or resume training**. By default it reads the existing test-prediction CSV,
recalculates metrics, displays charts and validation history, and saves results in a new timestamped folder.
It reconstructs results from saved artifacts, not the original missing console outputs.

Optional cells preserve the original post-training steps: load the best adapter, run fresh test inference,
and export a merged model. Set `RERUN_TEST_INFERENCE=True` only if new predictions are required.
The original model used a previous adapter merged into the base **plus** its Phase-2 adapter.
The fresh-inference path preserves this model loading, MediaPipe cache, prompt and dual-crop logic.

For saved results no GPU, PyTorch, MediaPipe, training images or model loading are required.
Dependencies: pandas, numpy, matplotlib, IPython.

**Save visible outputs:** use Jupyter's Run All then Save, or the `nbconvert` command below.
Do not use the previous bare `jupyter execute` command and assume outputs will be saved.

In [1]:
import os
import json
import time
import sys
import importlib.util
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage

RERUN_TEST_INFERENCE = False
CREATE_MERGED_FP32 = False
HOME = Path('/home/jovyan')
PROJECT_ROOT = HOME / 'EdgeVLM_finetune vlm256'
OUTPUT_ROOT = Path(os.environ.get('EDGEVLM_RESULTS_ROOT', str(
    PROJECT_ROOT / 'phase2_256m_mediapipe_targetaug_sqrt_150_per_class_patience3')))
PREDICTIONS_CSV = OUTPUT_ROOT / 'phase2_256m_all_techniques_mediapipe_test_predictions.csv'
BEST_ADAPTER_DIR = OUTPUT_ROOT / 'best_by_macro_f1'
TEST_CSV = HOME / 'phase2-tain-val-test-splits' / 'final_test.csv'
LABELS = ['NORMAL', 'DROWSY', 'LOOKING_AWAY', 'PHONE_USAGE']
EXPECTED_TEST_IMAGES = 5558
if not OUTPUT_ROOT.is_dir():
    raise FileNotFoundError(f'Results folder not found: {OUTPUT_ROOT}. Update OUTPUT_ROOT.')
REPORT_BASE = Path(os.environ.get('EDGEVLM_REPORT_BASE', str(OUTPUT_ROOT / 'evaluation_only_reports')))
REPORT_DIR = REPORT_BASE / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
REPORT_DIR.mkdir(parents=True, exist_ok=False)
print('Mode:', 'Fresh test inference' if RERUN_TEST_INFERENCE else 'Saved predictions; no training or inference')
print('Source:', OUTPUT_ROOT)
print('New reports:', REPORT_DIR)

Mode: Saved predictions; no training or inference
Source: /home/jovyan/EdgeVLM_finetune vlm256/phase2_256m_mediapipe_targetaug_sqrt_150_per_class_patience3
New reports: /home/jovyan/EdgeVLM_finetune vlm256/phase2_256m_mediapipe_targetaug_sqrt_150_per_class_patience3/evaluation_only_reports/20261018_183037_310200


## 1. Training history and early-stopping evidence

In [2]:
history_path = OUTPUT_ROOT / 'macro_f1_epoch_log.json'
if history_path.is_file():
    history = json.loads(history_path.read_text(encoding='utf-8'))
    history_df = pd.json_normalize(history)
    history_df.insert(0, 'evaluation_record', range(1, len(history_df) + 1))
    display(history_df)
    history_df.to_csv(REPORT_DIR / 'validation_history.csv', index=False)
    if len(history_df):
        best_i = history_df['macro_f1'].idxmax()
        print(f"Best recorded validation Macro-F1: {100*history_df.loc[best_i, 'macro_f1']:.2f}%")
        print(f"Best recorded epoch: {history_df.loc[best_i, 'epoch']:.4f}")
        print('Subsequent non-best records:', len(history_df) - 1 - best_i)
        duplicate_epoch = history_df['epoch'].round(3).duplicated(keep=False)
        if duplicate_epoch.any():
            print('Repeated epoch evaluations exist, likely from the resumed run. These are not extra epochs.')
        print('Original patience was 3. Epochs 3, 4 and 5 below the epoch-2 best support early stopping.')
        print('These validation scores are not full-test accuracy.')
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(history_df['evaluation_record'], history_df['macro_f1'] * 100, marker='o')
        ax.set_xticks(history_df['evaluation_record'])
        ax.set_xticklabels([f"{i}: epoch {e:.2f}" for i,e in zip(history_df.evaluation_record, history_df.epoch)], rotation=25, ha='right')
        ax.set(xlabel='Recorded evaluation (including repeated epochs)', ylabel='Validation Macro-F1 (%)', title='Validation history')
        ax.grid(alpha=.2)
        fig.tight_layout()
        fig.savefig(REPORT_DIR / 'validation_macro_f1.png', dpi=160)
        plt.show()
else:
    print('Validation history unavailable:', history_path)
progress = OUTPUT_ROOT / 'training_progress_history.log'
if progress.is_file():
    print('Last recorded training progress (not a live status):')
    print('\n'.join(progress.read_text(encoding='utf-8').splitlines()[-6:]))

   evaluation_record     epoch  macro_f1  improved     seconds  \
0                  1  0.999768  0.750013      True  145.705278   
1                  2  1.999846  0.763389      True  142.927300   
2                  3  2.999923  0.726050     False  140.594771   
3                  4  4.000000  0.751925     False  142.724277   
4                  5  4.999768  0.735605     False  143.102390   
5                  6  4.999768  0.699707     False  143.052779   

   per_class_f1.NORMAL  per_class_f1.DROWSY  per_class_f1.LOOKING_AWAY  \
0             0.700297             0.818991                   0.695341   
1             0.697548             0.809689                   0.735714   
2             0.642857             0.798680                   0.634538   
3             0.694737             0.767442                   0.767742   
4             0.656716             0.811388                   0.688889   
5             0.623810             0.816327                   0.598361   

   per_class_f1.PH

Best recorded validation Macro-F1: 76.34%
Best recorded epoch: 1.9998
Subsequent non-best records: 4
Repeated epoch evaluations exist, likely from the resumed run. These are not extra epochs.
Original patience was 3. Epochs 3, 4 and 5 below the epoch-2 best support early stopping.
These validation scores are not full-test accuracy.


<Figure size 900x400 with 1 Axes>

Last recorded training progress (not a live status):
 62.35% | step 16150/25904 | elapsed 26:23:48 | estimated remaining 15:56:33 | ETA 2026-10-19 06:14:42
 62.44% | step 16175/25904 | elapsed 26:26:07 | estimated remaining 15:54:01 | ETA 2026-10-19 06:14:29
Training started | 0.00% | step 16100/25904 | 2026-10-18 15:42:34
 62.25% | step 16125/25904 | elapsed 00:02:23 | estimated remaining 15:37:42 | ETA 2026-10-19 07:22:40
 62.35% | step 16150/25904 | elapsed 00:04:44 | estimated remaining 15:26:36 | ETA 2026-10-19 07:13:55
 62.44% | step 16175/25904 | elapsed 00:07:06 | estimated remaining 15:23:01 | ETA 2026-10-19 07:12:42


## 2. Optional: load the trained best model
Skipped by default. This is the equivalent of the original model-reload cell, with no training setup.
Fresh inference requires the same server files and original package environment.
The previous base adapter must still be available. The local caches are used without automatic installs/downloads.

In [3]:
if RERUN_TEST_INFERENCE or CREATE_MERGED_FP32:
    os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
    os.environ['USE_TF'] = '0'
    os.environ['TRANSFORMERS_NO_TF'] = '1'
    os.environ['USE_FLAX'] = '0'
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    import torch
    import transformers
    import tokenizers
    from transformers import AutoProcessor, AutoModelForVision2Seq
    from peft import PeftModel
    if not torch.cuda.is_available():
        raise RuntimeError('Fresh inference/model export requires CUDA in this notebook.')
    if transformers.__version__ != '4.46.3' or tokenizers.__version__ != '0.20.3':
        raise RuntimeError('Match the original environment: Transformers 4.46.3, Tokenizers 0.20.3.')
    device = torch.device('cuda:0')
    dtype = torch.float16
    IMAGE_SIZE = 512
    START_FROM_PREVIOUS_ADAPTER = True
    MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
    MODEL_CACHE_CANDIDATES = [
        PROJECT_ROOT / "model_cache",
        HOME / "EdgeVLM_finetune" / "model_cache",
    ]
    MODEL_CACHE = next((p for p in MODEL_CACHE_CANDIDATES if p.exists()), MODEL_CACHE_CANDIDATES[0])

    PREVIOUS_ADAPTER_CANDIDATES = [
        PROJECT_ROOT / "smolvlm_256m_lora_augmented_classweighted",
        HOME / "EdgeVLM_finetune_vlm256" / "smolvlm_256m_lora_augmented_classweighted",
        HOME / "EdgeVLM_finetune" / "smolvlm_256m_lora_augmented_classweighted",
    ]
    PREVIOUS_ADAPTER_DIR = next(
        (p for p in PREVIOUS_ADAPTER_CANDIDATES if (p / "adapter_model.safetensors").is_file()),
        None,
    )


    def load_plain_base():
        return AutoModelForVision2Seq.from_pretrained(
            MODEL_ID, cache_dir=str(MODEL_CACHE), torch_dtype=dtype,
            low_cpu_mem_usage=True, local_files_only=True)
    def load_starting_base():
        base = load_plain_base()
        if START_FROM_PREVIOUS_ADAPTER:
            if PREVIOUS_ADAPTER_DIR is None:
                raise FileNotFoundError(
                    "START_FROM_PREVIOUS_ADAPTER=True but the previous adapter was not found."
                )
            previous_model = PeftModel.from_pretrained(base, str(PREVIOUS_ADAPTER_DIR))
            base = previous_model.merge_and_unload()
            print("Merged previous good text-LoRA adapter into the 256M starting base.")
        else:
            print("Starting from the untouched pretrained 256M base.")
        return base

    if not (BEST_ADAPTER_DIR / 'adapter_model.safetensors').is_file():
        raise FileNotFoundError(f'Best adapter missing: {BEST_ADAPTER_DIR}')
    eval_processor = AutoProcessor.from_pretrained(str(BEST_ADAPTER_DIR), local_files_only=True)
    eval_base = load_starting_base()
    eval_model = PeftModel.from_pretrained(eval_base, str(BEST_ADAPTER_DIR)).to(device)
    eval_model.eval()
    eval_model.config.use_cache = True
    print('Loaded best adapter:', BEST_ADAPTER_DIR)
else:
    print("Optional model operation skipped; using saved test predictions.")

Optional model operation skipped; using saved test predictions.


## 3. Optional: reconstruct the original test preprocessing and prompt
Uses the saved MediaPipe boxes and blendshape cues. Requires cache coverage for every test image,
including records where face detection failed. It will not silently switch to different preprocessing.
No training augmentation is applied.

In [4]:
if RERUN_TEST_INFERENCE:
    import cv2
    from PIL import Image
    LANGGRAPH_MODULE_PATH = HOME / 'EdgeVLM_finetune' / 'langgraph_pipeline.py'
    if not LANGGRAPH_MODULE_PATH.is_file():
        raise FileNotFoundError(LANGGRAPH_MODULE_PATH)
    spec = importlib.util.spec_from_file_location('langgraph_pipeline', LANGGRAPH_MODULE_PATH)
    lgp = importlib.util.module_from_spec(spec)
    sys.modules['langgraph_pipeline'] = lgp
    spec.loader.exec_module(lgp)
    test_df = pd.read_csv(TEST_CSV)
    test_df['label'] = test_df['label'].astype(str).str.strip().str.upper()
    if not test_df.label.isin(LABELS).all():
        raise ValueError('Unexpected true labels in test manifest.')
    def resolve_image_path(value):
        path = Path(str(value).strip().replace("\\", "/"))
        return path if path.is_absolute() else HOME / path

    test_df['resolved_path'] = test_df.frame_path.map(lambda p: str(resolve_image_path(p)))
    if not all(Path(p).is_file() for p in test_df.resolved_path):
        raise FileNotFoundError('Some test images are missing.')
    MEDIAPIPE_CACHE = PROJECT_ROOT / 'phase2_256m_all_techniques_mediapipe_phone_prompt' / 'mediapipe_cache'
    cue_df = pd.read_csv(MEDIAPIPE_CACHE / 'face_blendshape_cues.csv')
    box_df = pd.read_csv(MEDIAPIPE_CACHE / 'face_landmark_boxes.csv')
    if cue_df.resolved_path.duplicated().any() or box_df.resolved_path.duplicated().any():
        raise ValueError('Duplicate cache keys; review before rerunning inference.')
    CUE_BY_PATH = dict(zip(cue_df.resolved_path.astype(str), cue_df.cue_text.astype(str)))
    BBOX_PROCESSED_PATHS = set(box_df.resolved_path.astype(str))
    FACE_BBOX_BY_PATH = {
        str(row.resolved_path): tuple(json.loads(row.face_bbox_json))
        for row in box_df.itertuples() if pd.notna(row.face_bbox_json) and str(row.face_bbox_json).strip()
    }
    paths = set(test_df.resolved_path)
    if not paths.issubset(CUE_BY_PATH) or not paths.issubset(BBOX_PROCESSED_PATHS):
        raise RuntimeError('Original MediaPipe cache coverage is incomplete. Restore the cache first.')
    def detect_live_face_bbox(rgb_image):
        raise RuntimeError('Unexpected uncached image; fresh results require the original cache.')
    def create_dual_crops(image_path):
        """Shared dual-crop preprocessing for training and all evaluation inference."""
        path = str(image_path)
        raw_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
        if raw_bgr is None:
            raise FileNotFoundError(f"OpenCV could not read: {image_path}")
        enhanced_rgb, frame_quality_ok = lgp.enhance_frame(raw_bgr)
        full_crop = lgp.resize_full_frame(enhanced_rgb)
        face_bbox = FACE_BBOX_BY_PATH.get(path)
        if path not in BBOX_PROCESSED_PATHS:
            face_bbox = detect_live_face_bbox(enhanced_rgb)
            BBOX_PROCESSED_PATHS.add(path)
            if face_bbox is not None:
                FACE_BBOX_BY_PATH[path] = face_bbox
        landmarks = {"face_bbox": face_bbox} if face_bbox is not None else None
        face_crop = lgp.extract_face_crop(enhanced_rgb, landmarks=landmarks)
        face_detected = face_bbox is not None or lgp.detect_face_bbox_haar(enhanced_rgb) is not None
        return full_crop, face_crop, bool(frame_quality_ok), bool(face_detected)
    INSTRUCTION = (
        "You are shown two images from the same in-cabin driver-camera frame. "
        "The first is the full available frame; the second is the face-focused crop. "
        "Use both images and the optional MediaPipe facial measurements below as "
        "additional visual evidence. The measurements are cues, not ground-truth labels "
        "or guaranteed diagnoses. Distinguish the two labels carefully: choose "
        "PHONE_USAGE when visible evidence shows the driver holding or interacting "
        "with a phone, even if the driver is also looking away. Choose LOOKING_AWAY "
        "when the driver's gaze or head is directed away from the road and there is "
        "no clear evidence of phone use. Do not infer phone use from gaze alone. Use "
        "the full frame for phone/hand context and the face crop for eye/head direction. "
        "Return exactly one label from NORMAL, DROWSY, LOOKING_AWAY, or PHONE_USAGE. "
        "Do not explain or add punctuation."
    )


    def instruction_with_cues(resolved_path):
        cue_text = CUE_BY_PATH.get(str(resolved_path), "unavailable (not cached)")
        return INSTRUCTION + "\n\nMediaPipe face cues (blendshape scores from 0 to 1): " + cue_text



    def parse_prediction(raw_text):
        normalized = str(raw_text).strip().upper()
        exact = normalized in LABELS
        cleaned = normalized.strip(" .,;:!?'\"")
        return (cleaned if cleaned in LABELS else "INVALID"), exact
    def predict_test_row(row):
        full_crop, face_crop, _, _ = create_dual_crops(row["resolved_path"])
        messages = [{
            "role": "user",
            "content": [
                {"type": "image"}, {"type": "image"},
                {"type": "text", "text": instruction_with_cues(row["resolved_path"])},
            ],
        }]
        prompt = eval_processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )
        inputs = eval_processor(
            text=prompt,
            images=[[Image.fromarray(full_crop), Image.fromarray(face_crop)]],
            return_tensors="pt",
            size={"longest_edge": IMAGE_SIZE},
            max_image_size={"longest_edge": IMAGE_SIZE},
        )
        inputs = {
            key: value.to(device) if torch.is_tensor(value) else value
            for key, value in inputs.items()
        }
        started = time.perf_counter()
        with torch.inference_mode():
            generated = eval_model.generate(
                **inputs, max_new_tokens=12, do_sample=False
            )
        latency_ms = (time.perf_counter() - started) * 1000
        input_length = inputs["input_ids"].shape[-1]
        raw = eval_processor.batch_decode(
            generated[:, input_length:], skip_special_tokens=True
        )[0].strip()
        prediction, exact = parse_prediction(raw)
        return raw, prediction, exact, latency_ms
else:
    print("Optional model operation skipped; using saved test predictions.")

Optional model operation skipped; using saved test predictions.


## 4. Test predictions
Default: read the completed original CSV. Optional fresh inference writes a **new** CSV inside this run's
report folder and leaves the original untouched. It shows progress and ETA, and writes
`evaluation_progress.txt` every 25 images. Timing follows the original generation-only method,
not complete camera-to-decision latency.

In [5]:
if RERUN_TEST_INFERENCE:
    from tqdm.auto import tqdm
    fresh_records = []
    started_all = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    for i, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Test inference'):
        raw, prediction, exact, latency_ms = predict_test_row(row)
        fresh_records.append(dict(row_index=int(i), frame_path=row['frame_path'], true_label=row['label'],
                                  raw_output=raw, prediction=prediction, exact_label_valid=exact, latency_ms=latency_ms))
        completed = len(fresh_records)
        if completed % 25 == 0 or completed == len(test_df):
            elapsed = time.perf_counter() - started_all
            remaining = elapsed / completed * (len(test_df) - completed)
            message = f'{completed}/{len(test_df)} ({100*completed/len(test_df):.2f}%) | remaining estimate {remaining/60:.1f} minutes'
            print(message, flush=True)
            (REPORT_DIR / 'evaluation_progress.txt').write_text(message, encoding='utf-8')
        if completed % 100 == 0 or completed == len(test_df):
            pd.DataFrame(fresh_records).to_csv(REPORT_DIR / 'fresh_test_predictions.csv', index=False)
    ACTIVE_PREDICTIONS_CSV = REPORT_DIR / 'fresh_test_predictions.csv'
    print('Current inference peak GPU allocation (MiB):', torch.cuda.max_memory_allocated()/1024**2)
else:
    ACTIVE_PREDICTIONS_CSV = PREDICTIONS_CSV
if not ACTIVE_PREDICTIONS_CSV.is_file():
    raise FileNotFoundError(f'Predictions not found: {ACTIVE_PREDICTIONS_CSV}')
predictions_df = pd.read_csv(ACTIVE_PREDICTIONS_CSV)
required = {'row_index', 'frame_path', 'true_label', 'raw_output', 'prediction', 'exact_label_valid', 'latency_ms'}
if required - set(predictions_df):
    raise ValueError(f'Missing columns: {required - set(predictions_df)}')
if predictions_df[list(required)].isna().any().any():
    raise ValueError('Missing values in required prediction fields.')
if predictions_df.row_index.duplicated().any() or predictions_df.frame_path.duplicated().any():
    raise ValueError('Duplicate row indices or image paths. Review rather than silently dropping rows.')
indices = pd.to_numeric(predictions_df.row_index, errors='raise').to_numpy()
if not np.array_equal(np.sort(indices), np.arange(EXPECTED_TEST_IMAGES)):
    raise ValueError(f'Expected exactly row indices 0..{EXPECTED_TEST_IMAGES-1}.')
predictions_df = predictions_df.sort_values('row_index').reset_index(drop=True)
if not predictions_df.true_label.isin(LABELS).all():
    raise ValueError('Unexpected true labels.')
if not predictions_df.prediction.isin(LABELS + ['INVALID']).all():
    raise ValueError('Unexpected prediction labels.')
exact_map = {'true': True, 'false': False, '1': True, '0': False}
exact = predictions_df.exact_label_valid.astype(str).str.strip().str.lower().map(exact_map)
if exact.isna().any():
    raise ValueError('Unrecognized exact-label validity values.')
raw_normalized = predictions_df.raw_output.astype(str).str.strip().str.upper()
if not np.array_equal(exact.to_numpy(), raw_normalized.isin(LABELS).to_numpy()):
    raise ValueError('Stored exact validity disagrees with raw outputs.')
parsed = raw_normalized.str.strip(' .,;:!?\'"')
parsed = parsed.where(parsed.isin(LABELS), 'INVALID')
if not np.array_equal(parsed.to_numpy(), predictions_df.prediction.to_numpy()):
    raise ValueError('Stored predictions disagree with the original parser.')
predictions_df['exact_label_valid'] = exact
latencies = pd.to_numeric(predictions_df.latency_ms, errors='raise').to_numpy()
if not np.isfinite(latencies).all() or (latencies < 0).any():
    raise ValueError('Invalid latency values.')
if TEST_CSV.is_file():
    manifest = pd.read_csv(TEST_CSV)
    if len(manifest) != len(predictions_df):
        raise ValueError('Prediction count differs from test manifest.')
    if not np.array_equal(manifest.frame_path.astype(str), predictions_df.frame_path.astype(str)):
        raise ValueError('Image order/paths differ from test manifest.')
    if not np.array_equal(manifest.label.astype(str).str.strip().str.upper(), predictions_df.true_label):
        raise ValueError('True labels differ from test manifest.')
    print('Verified image identities and labels against final_test.csv.')
else:
    print('Test manifest unavailable: internal completeness checked, external image/label mapping not verified.')
print('Validated prediction rows:', len(predictions_df))
display(predictions_df.head())
print('Source CSV:', ACTIVE_PREDICTIONS_CSV)
print('Saved CSV does not record the checkpoint hash; model provenance cannot be independently proved from CSV alone.')

Verified image identities and labels against final_test.csv.
Validated prediction rows: 5558


[Image-path preview omitted from the repository copy.]

Source CSV: /home/jovyan/EdgeVLM_finetune vlm256/phase2_256m_mediapipe_targetaug_sqrt_150_per_class_patience3/phase2_256m_all_techniques_mediapipe_test_predictions.csv
Saved CSV does not record the checkpoint hash; model provenance cannot be independently proved from CSV alone.


## 5. Final metrics and per-class results

In [6]:
y_true = predictions_df.true_label.to_numpy()
y_pred = predictions_df.prediction.to_numpy()
support = np.array([(y_true == label).sum() for label in LABELS])
predicted_counts = np.array([(y_pred == label).sum() for label in LABELS])
true_positive = np.array([((y_true == label) & (y_pred == label)).sum() for label in LABELS])
precision = np.divide(true_positive, predicted_counts, out=np.zeros(4), where=predicted_counts != 0)
recall = np.divide(true_positive, support, out=np.zeros(4), where=support != 0)
denominator = support + predicted_counts
f1 = np.divide(2*true_positive, denominator, out=np.zeros(4), where=denominator != 0)
unsafe_true = np.isin(y_true, LABELS[1:])
normal_true = y_true == 'NORMAL'
unsafe_pred = np.isin(y_pred, LABELS[1:])
metrics = {
    'images': int(len(predictions_df)),
    'correct_predictions': int((y_true == y_pred).sum()),
    'accuracy_percent': float(100 * np.mean(y_true == y_pred)),
    'balanced_accuracy_percent': float(100 * recall.mean()),
    'macro_f1_percent': float(100 * f1.mean()),
    'exact_label_validity_percent': float(100 * predictions_df.exact_label_valid.mean()),
    'parser_failure_percent': float(100 * np.mean(y_pred == 'INVALID')),
    'unsafe_to_safe_miss_rate_percent': float(100 * np.mean(np.isin(y_pred[unsafe_true], ['NORMAL', 'INVALID']))),
    'unsafe_class_error_rate_percent': float(100 * np.mean(y_pred[unsafe_true] != y_true[unsafe_true])),
    'false_alarm_rate_percent': float(100 * np.mean(unsafe_pred[normal_true])),
    'mean_latency_ms': float(latencies.mean()),
    'p95_latency_ms': float(np.percentile(latencies, 95)),
}
display(pd.DataFrame({'metric': list(metrics), 'value': list(metrics.values())}).round(2))
per_class = pd.DataFrame({'label': LABELS, 'precision_percent': precision*100, 'recall_percent': recall*100,
                          'f1_percent': f1*100, 'support': support})
display(per_class.round(2))
(REPORT_DIR / 'evaluation_only_test_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
per_class.to_csv(REPORT_DIR / 'per_class_metrics.csv', index=False)
print(f"Overall test accuracy: {metrics['accuracy_percent']:.2f}%")
print('Unsafe miss follows the original notebook: unsafe true labels predicted NORMAL or INVALID.')
print('False alarm follows the original notebook: NORMAL predicted one of the three unsafe labels.')
print('Latency is recorded generation-only timing, not end-to-end MediaPipe or Jetson latency.')
print('Original peak GPU memory cannot be reconstructed from this CSV.')

                              metric    value
0                             images  5558.00
1                correct_predictions  4779.00
2                   accuracy_percent    85.98
3          balanced_accuracy_percent    82.46
4                   macro_f1_percent    80.61
5       exact_label_validity_percent   100.00
6             parser_failure_percent     0.00
7   unsafe_to_safe_miss_rate_percent    13.85
8    unsafe_class_error_rate_percent    19.64
9           false_alarm_rate_percent    10.10
10                   mean_latency_ms   306.67
11                    p95_latency_ms   429.38

          label  precision_percent  recall_percent  f1_percent  support
0        NORMAL              90.31           89.90       90.11     3277
1        DROWSY              76.14           85.01       80.33     1021
2  LOOKING_AWAY              57.18           79.14       66.39      302
3   PHONE_USAGE              98.37           75.78       85.61      958

Overall test accuracy: 85.98%
Unsafe miss follows the original notebook: unsafe true labels predicted NORMAL or INVALID.
False alarm follows the original notebook: NORMAL predicted one of the three unsafe labels.
Latency is recorded generation-only timing, not end-to-end MediaPipe or Jetson latency.
Original peak GPU memory cannot be reconstructed from this CSV.


## 6. Confusion matrix, class scores and latency charts

In [7]:
matrix_labels = LABELS + (['INVALID'] if (y_pred == 'INVALID').any() else [])
cm = np.array([[((y_true == true) & (y_pred == pred)).sum() for pred in matrix_labels] for true in matrix_labels])
cm_df = pd.DataFrame(cm, index=matrix_labels, columns=matrix_labels)
cm_df.index.name = 'True label'
cm_df.columns.name = 'Predicted label'
display(cm_df)
cm_df.to_csv(REPORT_DIR / 'confusion_matrix.csv')
fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(cm, cmap='Blues')
fig.colorbar(image, ax=ax, label='Images')
ax.set_xticks(range(len(matrix_labels)), matrix_labels, rotation=30, ha='right')
ax.set_yticks(range(len(matrix_labels)), matrix_labels)
for i in range(len(matrix_labels)):
    for j in range(len(matrix_labels)):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center', color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set(xlabel='Predicted label', ylabel='True label', title='MediaPipe test confusion matrix')
fig.tight_layout()
fig.savefig(REPORT_DIR / 'confusion_matrix.png', dpi=160)
plt.show()
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(LABELS))
for shift, name, values in [(-.25, 'Precision', precision), (0, 'Recall', recall), (.25, 'F1', f1)]:
    ax.bar(x+shift, values*100, width=.25, label=name)
ax.set_xticks(x, LABELS)
ax.set(ylim=(0, 105), ylabel='Percent', title='Per-class test performance')
ax.legend(loc='lower left')
fig.tight_layout()
fig.savefig(REPORT_DIR / 'per_class_scores.png', dpi=160)
plt.show()
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(latencies, bins=35, color='#236B8E')
ax.axvline(np.percentile(latencies,95), color='#B45309', linestyle='--', label='95th percentile')
ax.set(xlabel='Recorded generation latency (ms)', ylabel='Images', title='Generation latency, excluding preprocessing')
ax.legend()
fig.tight_layout()
fig.savefig(REPORT_DIR / 'latency_histogram.png', dpi=160)
plt.show()

Predicted label  NORMAL  DROWSY  LOOKING_AWAY  PHONE_USAGE
True label                                                
NORMAL             2946     265            58            8
DROWSY              148     868             5            0
LOOKING_AWAY         57       2           239            4
PHONE_USAGE         111       5           116          726

<Figure size 800x600 with 2 Axes>

<Figure size 900x400 with 1 Axes>

<Figure size 900x400 with 1 Axes>

## 7. Misclassification review and saved crop preview

In [8]:
errors = predictions_df.loc[predictions_df.true_label != predictions_df.prediction].copy()
errors.to_csv(REPORT_DIR / 'test_misclassification_review.csv', index=False)
confusions = errors.groupby(['true_label', 'prediction']).size().sort_values(ascending=False).rename('images').reset_index()
print('Incorrect predictions:', len(errors))
display(confusions)
display(errors.head(20))
preview = OUTPUT_ROOT / 'preprocessing_sanity_check.png'
if preview.is_file():
    print('Saved preprocessing preview from the original run:')
    display(DisplayImage(filename=str(preview), width=900))
else:
    print('Saved preprocessing preview not available.')

Incorrect predictions: 779


      true_label    prediction  images
0         NORMAL        DROWSY     265
1         DROWSY        NORMAL     148
2    PHONE_USAGE  LOOKING_AWAY     116
3    PHONE_USAGE        NORMAL     111
4         NORMAL  LOOKING_AWAY      58
5   LOOKING_AWAY        NORMAL      57
6         NORMAL   PHONE_USAGE       8
7    PHONE_USAGE        DROWSY       5
8         DROWSY  LOOKING_AWAY       5
9   LOOKING_AWAY   PHONE_USAGE       4
10  LOOKING_AWAY        DROWSY       2

[Image-path preview omitted from the repository copy.]

Saved preprocessing preview from the original run:


<IPython.core.display.Image object>

## 8. Optional merged FP32 deployment export
Disabled by default, as in the original notebook. If enabled in the configuration cell, it saves to the new
report folder only. This export does not establish Jetson compatibility or runtime performance.

In [9]:
if CREATE_MERGED_FP32:
    MERGED_DIR = REPORT_DIR / 'merged_model_256m_mediapipe_fp32'
    merged_model = eval_model.merge_and_unload().float()
    merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
    eval_processor.save_pretrained(str(MERGED_DIR))
    print('Merged FP32 model:', MERGED_DIR)
else:
    print("Optional model operation skipped; using saved test predictions.")

Optional model operation skipped; using saved test predictions.


In [10]:
summary = {
    'completed_at': datetime.now().isoformat(),
    'mode': 'fresh_test_inference' if RERUN_TEST_INFERENCE else 'saved_prediction_analysis',
    'prediction_source': str(ACTIVE_PREDICTIONS_CSV),
    'report_directory': str(REPORT_DIR),
    'training_executed': False,
    'images': len(predictions_df),
    'accuracy_percent': metrics['accuracy_percent'],
}
(REPORT_DIR / 'evaluation_complete.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('RESULTS NOTEBOOK COMPLETED. No training was run.')
print('Reports saved in:', REPORT_DIR)
print('Save this notebook now if running interactively.')

RESULTS NOTEBOOK COMPLETED. No training was run.
Reports saved in: /home/jovyan/EdgeVLM_finetune vlm256/phase2_256m_mediapipe_targetaug_sqrt_150_per_class_patience3/evaluation_only_reports/20261018_183037_310200
Save this notebook now if running interactively.


## Terminal execution with saved cell outputs
Upload this notebook to `/home/jovyan/phase2_fine tune/Mediapipe`, then run:

```bash
cd "/home/jovyan/phase2_fine tune/Mediapipe"
jupyter nbconvert --to notebook --execute "V3_EdgeVLM_256M_MediaPipe_PostTraining_Results.ipynb" --output "V3_EdgeVLM_256M_MediaPipe_PostTraining_Results_executed.ipynb" --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=python3
```

Download the `_executed.ipynb` file after the command finishes. It contains the visible tables and figures.
Default saved-CSV analysis should be much faster than fresh model inference. No training GPU is needed.